In [ ]:
print("Hello")

Hello


In [ ]:
#!/usr/bin/env python3
"""
Azure DevOps Pipeline Monitor
------------------------------
Checks whether any pipelines are currently running (queued or in-progress)
in a given Azure DevOps project, and sends an email alert if none are running.

Recommended usage: run this script every 30 minutes via cron (Linux/Mac) or
Task Scheduler (Windows). A single-check design like this is more reliable
than a long-running Python loop, because the OS scheduler will keep re-running
it even if the machine reboots or the previous run crashed.

If you'd rather have the script loop forever on its own, pass --loop.

------------------------------------------------------------------------------
SETUP
------------------------------------------------------------------------------
1. Create an Azure DevOps Personal Access Token (PAT):
   Azure DevOps -> User Settings -> Personal Access Tokens -> New Token
   Scope needed: "Build (Read)"  (Read-only is enough)

2. Set the following environment variables (don't hardcode secrets in the script):
   Linux/Mac:
       export AZDO_PAT="your_pat_here"
       export SMTP_PASSWORD="your_email_app_password"

   Windows (PowerShell):
       setx AZDO_PAT "your_pat_here"
       setx SMTP_PASSWORD "your_email_app_password"

3. Edit the CONFIG section below (org, project, email settings).

4. Install dependency:
       pip install requests

5. Test it manually:
       python check_pipelines.py

6. Schedule it:
   Linux/Mac (crontab -e):
       */30 * * * * /usr/bin/python3 /path/to/check_pipelines.py >> /path/to/pipeline_monitor.log 2>&1

   Windows Task Scheduler:
       Create a Basic Task -> Trigger: Repeat every 30 minutes ->
       Action: Start a program -> python.exe -> Arguments: C:\path\to\check_pipelines.py
------------------------------------------------------------------------------
"""

import os
import sys
import json
import time
import base64
import logging
import smtplib
import argparse
from email.mime.text import MIMEText
from datetime import datetime, timezone

import requests

# ============================== CONFIG =====================================

ORGANIZATION = "1Wan"                 # from your URL: dev.azure.com/1Wan
PROJECT = "SWAN"                      # from your URL: /SWAN/_build

# Optional: limit the check to specific pipeline definitions (by name).
# Leave as None to check ALL pipelines in the project.
PIPELINE_NAME_FILTER = None           # e.g. ["WAN Chaos Test Pipeline", "wan-oob-nightly"]

# Email (SMTP) settings
SMTP_SERVER = "smtp.office365.com"    # e.g. smtp.gmail.com, smtp.office365.com
SMTP_PORT = 587
SMTP_USERNAME = "your_email@company.com"
EMAIL_FROM = "your_email@company.com"
EMAIL_TO = ["you@company.com", "teammate@company.com"]
EMAIL_SUBJECT = "[Alert] No Azure DevOps pipelines currently running"

# Avoid re-sending an email every single 30-min check while the "no pipelines
# running" condition persists. Only re-alert after this many minutes.
RE_ALERT_INTERVAL_MINUTES = 120

# Where to store "last alert sent" state so repeated runs know not to spam you
STATE_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "pipeline_monitor_state.json")

LOG_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "pipeline_monitor.log")

# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)


def get_pat():
    pat = os.environ.get("AZDO_PAT")
    if not pat:
        log.error("AZDO_PAT environment variable is not set. See setup instructions.")
        sys.exit(1)
    return pat


def get_auth_header(pat):
    token = base64.b64encode(f":{pat}".encode()).decode()
    return {"Authorization": f"Basic {token}"}


def get_running_builds(pat):
    """
    Queries Azure DevOps for builds that are currently in progress or not yet started.
    Docs: https://learn.microsoft.com/en-us/rest/api/azure/devops/build/builds/list
    """
    url = f"https://dev.azure.com/{ORGANIZATION}/{PROJECT}/_apis/build/builds"
    params = {
        "statusFilter": "inProgress,notStarted",
        "api-version": "7.1",
    }
    headers = get_auth_header(pat)

    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    builds = data.get("value", [])

    if PIPELINE_NAME_FILTER:
        builds = [b for b in builds if b.get("definition", {}).get("name") in PIPELINE_NAME_FILTER]

    return builds


def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f:
                return json.load(f)
        except (json.JSONDecodeError, IOError):
            return {}
    return {}


def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f)


def should_send_alert(state):
    last_alert = state.get("last_alert_utc")
    if not last_alert:
        return True
    last_alert_dt = datetime.fromisoformat(last_alert)
    minutes_since = (datetime.now(timezone.utc) - last_alert_dt).total_seconds() / 60
    return minutes_since >= RE_ALERT_INTERVAL_MINUTES


def send_email_alert():
    smtp_password = os.environ.get("SMTP_PASSWORD")
    if not smtp_password:
        log.error("SMTP_PASSWORD environment variable is not set. Cannot send email.")
        return False

    body = (
        f"No Azure DevOps pipelines are currently running or queued.\n\n"
        f"Organization: {ORGANIZATION}\n"
        f"Project: {PROJECT}\n"
        f"Checked at (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}\n\n"
        f"View pipelines: https://dev.azure.com/{ORGANIZATION}/{PROJECT}/_build?view=pipelines\n"
    )

    msg = MIMEText(body)
    msg["Subject"] = EMAIL_SUBJECT
    msg["From"] = EMAIL_FROM
    msg["To"] = ", ".join(EMAIL_TO)

    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT, timeout=30) as server:
            server.starttls()
            server.login(SMTP_USERNAME, smtp_password)
            server.sendmail(EMAIL_FROM, EMAIL_TO, msg.as_string())
        log.info("Alert email sent to %s", EMAIL_TO)
        return True
    except Exception as e:
        log.error("Failed to send email: %s", e)
        return False


def run_check():
    pat = get_pat()
    try:
        running_builds = get_running_builds(pat)
    except requests.exceptions.RequestException as e:
        log.error("Failed to query Azure DevOps API: %s", e)
        return

    if running_builds:
        names = ", ".join(b.get("definition", {}).get("name", "unknown") for b in running_builds)
        log.info("%d pipeline(s) running/queued: %s", len(running_builds), names)
        # Reset alert state so the next "no pipelines running" event alerts immediately
        save_state({})
        return

    log.warning("No pipelines currently running or queued.")
    state = load_state()
    if should_send_alert(state):
        if send_email_alert():
            state["last_alert_utc"] = datetime.now(timezone.utc).isoformat()
            save_state(state)
    else:
        log.info("Skipping email — already alerted within the last %d minutes.", RE_ALERT_INTERVAL_MINUTES)


def main():
    parser = argparse.ArgumentParser(description="Monitor Azure DevOps pipelines and email if none are running.")
    parser.add_argument("--loop", action="store_true", help="Run continuously, checking every 30 minutes.")
    parser.add_argument("--interval", type=int, default=30, help="Minutes between checks in --loop mode (default 30).")
    args = parser.parse_args()

    if args.loop:
        log.info("Starting continuous monitor loop (every %d minutes). Press Ctrl+C to stop.", args.interval)
        while True:
            run_check()
            time.sleep(args.interval * 60)
    else:
        run_check()


if __name__ == "__main__":
    main()

A quick note first: scraping that web page isn't the way to go — Azure DevOps has a REST API that gives you clean, reliable pipeline status data (no HTML parsing, no login-wall issues). I'll build the script around that.

Here's my plan, then the script:

- **Auth**: Use a Personal Access Token (PAT) with "Build (Read)" scope — put it in an environment variable, never hardcode it.
- **Check logic**: Call the Builds API filtered by `statusFilter=inProgress`. If the count is 0, send an email.
- **Scheduling**: I'd recommend running this as a single check via **cron** (Linux) or **Task Scheduler** (Windows) every 30 minutes, rather than a Python script with an infinite `while True: sleep(1800)` loop. A scheduler is more robust — if the machine reboots or the script crashes, cron/Task Scheduler will just re-run it, whereas a sleeping loop dies silently. I've included both options in the script (a `--loop` flag if you want the loop version anyway).
- **Alert fatigue**: I added a simple "don't re-notify every 30 min if it's still empty" option (state file) — otherwise you'll get spammed with emails during normal lulls (e.g., overnight).**How it works**

1. Calls the Azure DevOps Builds REST API filtered to `inProgress,notStarted` builds for your `1Wan` org / `SWAN` project (pulled straight from your URL).
2. If the list is empty → sends an email via SMTP.
3. Won't spam you — it only re-alerts every 2 hours (configurable) while the "nothing running" condition persists, resetting once a pipeline starts again.
4. Logs everything to `pipeline_monitor.log` next to the script, so you can check history/debug failures.

**What you need to fill in / do**

| Item | Where |
|---|---|
| PAT (Build - Read scope) | `export AZDO_PAT="..."` |
| Email account password/app-password | `export SMTP_PASSWORD="..."` |
| SMTP server, from/to addresses | top `CONFIG` section |
| Optionally restrict to specific pipelines | `PIPELINE_NAME_FILTER` |

**Scheduling — my recommendation:** don't run it as an infinite Python loop in a terminal window. Use **cron** (Linux/Mac) or **Task Scheduler** (Windows) to run the script every 30 minutes instead — it survives reboots/crashes automatically, unlike a sleeping script. Both are documented in the file's docstring. If you still want the self-looping version for a quick test, run `python check_pipelines.py --loop`.

**Other suggestions**

- If you have a **Slack/Teams webhook** already set up for the team, that's often more effective than email for this kind of "pipelines are idle" alert (faster, and less likely to be missed) — happy to add a webhook version if you want.
- You could also flip the condition: alert when a pipeline **fails** (red ❌ like "WAN Audit Log Heartbeat" or "SWAN Saturday 8 AM PST" in your screenshot) rather than when nothing is running — often more actionable. I can add that as a second check if useful.
- Store the PAT in a proper secrets manager (Azure Key Vault, etc.) if this runs on a shared server rather than your laptop.

Good additions — here's the updated script with both changes:

1. **Teams webhook notification** alongside email (fires both every time).
2. **Per-pipeline monitoring** — instead of just "is *anything* running," you give it a list of specific pipeline names to track, and it tells you exactly *which* of them have no active run (much more useful than the previous "any pipeline" check).That warning is harmless — it's just from a Windows path (`C:\path\to\...`) inside the docstring, not real code.**What changed**

1. **Teams webhook**
   - In Teams: open the channel → **`...`** → **Connectors** → **Incoming Webhook** → **Create**, give it a name, copy the URL. (Some newer Microsoft 365 tenants have replaced this with **Workflows** → *"Post to a channel when a webhook request is received"* — that gives an equivalent URL and works the same way with this script.)
   - Set it as `TEAMS_WEBHOOK_URL` env var (or paste directly into the config).
   - `ENABLE_EMAIL = True` / `ENABLE_TEAMS = True` — flip either off if you only want one channel later.

2. **Multi-pipeline monitoring**
   - `MONITORED_PIPELINES` is now a list of exact pipeline names (I pre-filled it with the ones visible in your screenshot — edit to match your real set).
   - The script pulls the names from each pipeline's most recent active build and figures out which of your named pipelines have **no** active run, then names them specifically in the alert (e.g. *"WAN Chaos Test Pipeline, wan-oob-nightly are idle"*) instead of a generic "nothing is running."
   - Leave `MONITORED_PIPELINES = []` to fall back to the old "alert if the *entire project* has zero active builds" behavior.

3. **Smarter re-alerting**: it now re-notifies immediately if the *set* of idle pipelines changes (e.g., a new one goes idle), not just on the time interval — so you're not stuck waiting 2 hours to hear about a fresh problem.

One thing worth deciding: do you want an alert only when a pipeline **stays idle continuously**, or is a single 30-min snapshot with nothing running enough (some pipelines legitimately only run nightly/weekly, so they'll look "idle" most of the day)? If it's the former, I can add per-pipeline schedule awareness so you don't get false alarms on pipelines that aren't supposed to be running right now.

In [ ]:
#!/usr/bin/env python3
"""
Azure DevOps Pipeline Monitor
------------------------------
Tracks a list of specific Azure DevOps pipelines and sends an alert
(Email and/or Microsoft Teams) whenever one of them currently has no
running or queued build.

Recommended usage: run this script every 30 minutes via cron (Linux/Mac) or
Task Scheduler (Windows). A single-check design like this is more reliable
than a long-running Python loop, because the OS scheduler will keep re-running
it even if the machine reboots or the previous run crashed.

If you'd rather have the script loop forever on its own, pass --loop.

------------------------------------------------------------------------------
SETUP
------------------------------------------------------------------------------
1. Create an Azure DevOps Personal Access Token (PAT):
   Azure DevOps -> User Settings -> Personal Access Tokens -> New Token
   Scope needed: "Build (Read)"  (Read-only is enough)

2. Create a Microsoft Teams "Incoming Webhook" connector on the channel you
   want alerts posted to:
   Teams channel -> "..." -> Connectors -> Incoming Webhook -> Create
   -> copy the webhook URL it gives you.
   (Newer tenants may require a "Workflow" instead of the classic Incoming
   Webhook connector -- Teams -> Workflows -> "Post to a channel when a
   webhook request is received" gives you an equivalent URL.)

3. Set the following environment variables (don't hardcode secrets in the script):
   Linux/Mac:
       export AZDO_PAT="your_pat_here"
       export SMTP_PASSWORD="your_email_app_password"
       export TEAMS_WEBHOOK_URL="https://.../IncomingWebhook/..."

   Windows (PowerShell):
       setx AZDO_PAT "your_pat_here"
       setx SMTP_PASSWORD "your_email_app_password"
       setx TEAMS_WEBHOOK_URL "https://.../IncomingWebhook/..."

4. Edit the CONFIG section below (org, project, pipelines to watch, email settings,
   and which notification channels to enable).

5. Install dependency:
       pip install requests

6. Test it manually:
       python check_pipelines.py

7. Schedule it:
   Linux/Mac (crontab -e):
       */30 * * * * /usr/bin/python3 /path/to/check_pipelines.py >> /path/to/pipeline_monitor.log 2>&1

   Windows Task Scheduler:
       Create a Basic Task -> Trigger: Repeat every 30 minutes ->
       Action: Start a program -> python.exe -> Arguments: C:\path\to\check_pipelines.py
------------------------------------------------------------------------------
"""

import os
import sys
import json
import time
import base64
import logging
import smtplib
import argparse
from email.mime.text import MIMEText
from datetime import datetime, timezone

import requests

# ============================== CONFIG =====================================

ORGANIZATION = "1Wan"                 # from your URL: dev.azure.com/1Wan
PROJECT = "SWAN"                      # from your URL: /SWAN/_build

# List every pipeline you want individually monitored, by exact name
# (must match the "Pipeline" column name in the Azure DevOps UI).
# Leave as an empty list [] to instead treat "at least one pipeline running
# anywhere in the project" as healthy (the old behavior).
MONITORED_PIPELINES = [
    "Update OS Versions in Inventory",
    "ITE_Weekly_E2E_Test",
    "WAN Audit Log Heartbeat",
    "WAN Chaos Test Pipeline",
    "wan-oob-nightly",
    "Global SDL - TSA",
    "wan_test_pr_validation",
    "wan-tk5-nightly",
    "WAN Test Pipeline",
]

# --- Notification channels: turn each on/off independently ---
ENABLE_EMAIL = True
ENABLE_TEAMS = True

# Email (SMTP) settings
SMTP_SERVER = "smtp.office365.com"    # e.g. smtp.gmail.com, smtp.office365.com
SMTP_PORT = 587
SMTP_USERNAME = "your_email@company.com"
EMAIL_FROM = "your_email@company.com"
EMAIL_TO = ["you@company.com", "teammate@company.com"]
EMAIL_SUBJECT = "[Alert] Azure DevOps pipeline(s) not running"

# Teams webhook URL can also just be hardcoded here instead of the env var
TEAMS_WEBHOOK_URL = os.environ.get("TEAMS_WEBHOOK_URL", "")

# Avoid re-sending an alert every single 30-min check while the same
# pipeline(s) stay idle. Only re-alert after this many minutes.
RE_ALERT_INTERVAL_MINUTES = 120

# Where to store "last alert sent" state so repeated runs know not to spam you
STATE_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "pipeline_monitor_state.json")

LOG_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), "pipeline_monitor.log")

# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(LOG_FILE), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)


def get_pat():
    pat = os.environ.get("AZDO_PAT")
    if not pat:
        log.error("AZDO_PAT environment variable is not set. See setup instructions.")
        sys.exit(1)
    return pat


def get_auth_header(pat):
    token = base64.b64encode(f":{pat}".encode()).decode()
    return {"Authorization": f"Basic {token}"}


def get_active_builds(pat):
    """
    Queries Azure DevOps for builds that are currently in progress or queued.
    Docs: https://learn.microsoft.com/en-us/rest/api/azure/devops/build/builds/list
    """
    url = f"https://dev.azure.com/{ORGANIZATION}/{PROJECT}/_apis/build/builds"
    params = {
        "statusFilter": "inProgress,notStarted",
        "api-version": "7.1",
    }
    headers = get_auth_header(pat)

    resp = requests.get(url, headers=headers, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json().get("value", [])


def find_idle_pipelines(active_builds):
    """
    Returns the subset of MONITORED_PIPELINES that currently have NO
    active (running/queued) build. If MONITORED_PIPELINES is empty,
    falls back to "the whole project has zero active builds" mode.
    """
    active_names = {b.get("definition", {}).get("name") for b in active_builds}

    if not MONITORED_PIPELINES:
        return [] if active_builds else ["<any pipeline in project>"]

    return [name for name in MONITORED_PIPELINES if name not in active_names]


def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f:
                return json.load(f)
        except (json.JSONDecodeError, IOError):
            return {}
    return {}


def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f)


def should_send_alert(state, idle_pipelines):
    """
    Re-alert if: we've never alerted, OR the set of idle pipelines has
    changed since the last alert, OR enough time has passed.
    """
    last_alert = state.get("last_alert_utc")
    last_idle_set = set(state.get("last_idle_pipelines", []))

    if not last_alert:
        return True
    if set(idle_pipelines) != last_idle_set:
        return True

    last_alert_dt = datetime.fromisoformat(last_alert)
    minutes_since = (datetime.now(timezone.utc) - last_alert_dt).total_seconds() / 60
    return minutes_since >= RE_ALERT_INTERVAL_MINUTES


def build_message(idle_pipelines):
    checked_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    pipeline_list = "\n".join(f"  - {name}" for name in idle_pipelines)
    text = (
        f"The following Azure DevOps pipeline(s) have no run currently "
        f"in progress or queued:\n\n{pipeline_list}\n\n"
        f"Organization: {ORGANIZATION}\n"
        f"Project: {PROJECT}\n"
        f"Checked at: {checked_at}\n\n"
        f"View pipelines: https://dev.azure.com/{ORGANIZATION}/{PROJECT}/_build?view=pipelines"
    )
    return text


def send_email_alert(message_text):
    smtp_password = os.environ.get("SMTP_PASSWORD")
    if not smtp_password:
        log.error("SMTP_PASSWORD environment variable is not set. Cannot send email.")
        return False

    msg = MIMEText(message_text)
    msg["Subject"] = EMAIL_SUBJECT
    msg["From"] = EMAIL_FROM
    msg["To"] = ", ".join(EMAIL_TO)

    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT, timeout=30) as server:
            server.starttls()
            server.login(SMTP_USERNAME, smtp_password)
            server.sendmail(EMAIL_FROM, EMAIL_TO, msg.as_string())
        log.info("Alert email sent to %s", EMAIL_TO)
        return True
    except Exception as e:
        log.error("Failed to send email: %s", e)
        return False


def send_teams_alert(idle_pipelines):
    if not TEAMS_WEBHOOK_URL:
        log.error("TEAMS_WEBHOOK_URL is not set. Cannot send Teams notification.")
        return False

    checked_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    pipeline_list_md = "\n".join(f"- {name}" for name in idle_pipelines)

    # Simple, widely-compatible "MessageCard" payload for Incoming Webhooks.
    # (If your tenant only offers the newer Workflows-based webhook, it also
    # accepts this same JSON shape for a basic text post.)
    payload = {
        "@type": "MessageCard",
        "@context": "http://schema.org/extensions",
        "summary": EMAIL_SUBJECT,
        "themeColor": "D9534F",
        "title": "⚠️ Azure DevOps pipeline(s) not running",
        "sections": [
            {
                "activityTitle": f"**Project:** {ORGANIZATION}/{PROJECT}",
                "text": f"**Idle pipelines:**\n\n{pipeline_list_md}\n\n**Checked at:** {checked_at}",
            }
        ],
        "potentialAction": [
            {
                "@type": "OpenUri",
                "name": "View Pipelines",
                "targets": [
                    {
                        "os": "default",
                        "uri": f"https://dev.azure.com/{ORGANIZATION}/{PROJECT}/_build?view=pipelines",
                    }
                ],
            }
        ],
    }

    try:
        resp = requests.post(TEAMS_WEBHOOK_URL, json=payload, timeout=30)
        resp.raise_for_status()
        log.info("Teams notification sent.")
        return True
    except requests.exceptions.RequestException as e:
        log.error("Failed to send Teams notification: %s", e)
        return False


def send_notifications(idle_pipelines):
    message_text = build_message(idle_pipelines)
    email_ok = True
    teams_ok = True

    if ENABLE_EMAIL:
        email_ok = send_email_alert(message_text)
    if ENABLE_TEAMS:
        teams_ok = send_teams_alert(idle_pipelines)

    return email_ok and teams_ok


def run_check():
    pat = get_pat()
    try:
        active_builds = get_active_builds(pat)
    except requests.exceptions.RequestException as e:
        log.error("Failed to query Azure DevOps API: %s", e)
        return

    idle_pipelines = find_idle_pipelines(active_builds)

    if not idle_pipelines:
        log.info("All monitored pipelines have an active run (or project-wide check found activity).")
        save_state({})
        return

    log.warning("Idle pipelines detected: %s", ", ".join(idle_pipelines))
    state = load_state()

    if should_send_alert(state, idle_pipelines):
        if send_notifications(idle_pipelines):
            save_state({
                "last_alert_utc": datetime.now(timezone.utc).isoformat(),
                "last_idle_pipelines": idle_pipelines,
            })
    else:
        log.info("Skipping alert — already notified about this within the last %d minutes.", RE_ALERT_INTERVAL_MINUTES)


def main():
    parser = argparse.ArgumentParser(description="Monitor Azure DevOps pipelines and alert via Email/Teams if idle.")
    parser.add_argument("--loop", action="store_true", help="Run continuously, checking every 30 minutes.")
    parser.add_argument("--interval", type=int, default=30, help="Minutes between checks in --loop mode (default 30).")
    args = parser.parse_args()

    if args.loop:
        log.info("Starting continuous monitor loop (every %d minutes). Press Ctrl+C to stop.", args.interval)
        while True:
            run_check()
            time.sleep(args.interval * 60)
    else:
        run_check()


if __name__ == "__main__":
    main()